In [0]:
dbutils.widgets.text("catalog", "fk_1361")
catalog = dbutils.widgets.get("catalog")

schema_path = f"/Volumes/{catalog}/ops/pipeline_state/schema/orders"
checkpoint_path = f"/Volumes/{catalog}/ops/pipeline_state/checkpoint/orders"
source_path = f"/Volumes/{catalog}/landing/raw/orders/"

In [0]:
%python
from pyspark.sql import functions as F
df = spark.readStream.format('cloudFiles') \
    .option('cloudFiles.format','json') \
    .option('cloudFiles.inferColumnTypes','true') \
    .option('cloudFiles.schemaLocation', schema_path) \
    .option('cloudFiles.schemaEvolutionMode', 'addNewColumns') \
    .option('multiline','true') \
    .load(source_path)

df = df.withColumn('ingestion_timestamp', F.current_timestamp())

df.writeStream.format('delta') \
    .option('checkpointLocation', checkpoint_path) \
    .trigger(availableNow=True) \
    .toTable(f'{catalog}.bronze.orders_bronze')

In [0]:
%sql
drop table fk_1361.bronze.orders_bronze